# Chapitre 4 - Méthodes linéaires de classification
## Cours 1 : régression sur indicatrices, LDA et QDA

Ce premier notebook couvre les sections 4.1 à 4.3.2 du livre :

1. le cadre de la classification et la notion de **frontière de décision linéaire** ;
2. la **régression linéaire sur la matrice indicatrice** et le phénomène de **masquage**
   (figures 4.2 et 4.3 du livre) ;
3. l'**analyse discriminante linéaire** (LDA) déduite du modèle gaussien et de la formule
   de Bayes (figure 4.5) ;
4. l'**analyse discriminante quadratique** (QDA) et sa version régularisée (RDA)
   (figures 4.6 et 4.7) ;
5. la comparaison de ces méthodes sur les données de voyelles (tableau 4.1).

La LDA de rang réduit fait l'objet du cours 2, la régression logistique et les hyperplans
séparateurs du cours 3.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch04_classification as c4
set_style()

## 1. Le problème de classification

On observe des couples $(x_i, g_i)$, $i = 1, \dots, N$, où $x_i \in \mathbb{R}^p$ est un
vecteur de prédicteurs et $g_i \in \mathcal{G} = \{1, \dots, K\}$ une **classe**. Un
classifieur est une fonction $\hat G : \mathbb{R}^p \to \mathcal{G}$ ; il découpe l'espace
des entrées en $K$ **régions de décision**, séparées par des **frontières de décision**.

Dans ce chapitre, les frontières sont **linéaires** : ce sont des morceaux d'hyperplans
affines $\{x : \beta_0 + \beta^T x = 0\}$.

Le cas typique : on dispose d'une **fonction discriminante** $\delta_k(x)$ par classe et on
classe vers la plus grande,
$$\hat G(x) = \operatorname*{arg\,max}_{k} \delta_k(x).$$
La frontière entre les classes $k$ et $\ell$ est $\{x : \delta_k(x) = \delta_\ell(x)\}$ ;
si chaque $\delta_k$ est affine en $x$, $\delta_k(x) = \beta_{k0} + \beta_k^T x$, alors cette
frontière est l'hyperplan $\{x : (\beta_{k0} - \beta_{\ell 0}) + (\beta_k - \beta_\ell)^T x = 0\}$.

> **Remarque.** Il suffit en fait qu'une **transformation monotone** de $\delta_k$ (ou de
> $\Pr(G = k \mid X = x)$) soit affine. Par exemple, pour la régression logistique à deux
> classes, c'est le logarithme du rapport des chances $\log \frac{p}{1-p}$ qui est affine
> (éq. (4.1)-(4.2) du livre).

> **Intuition.** On peut aussi obtenir des frontières *non linéaires* avec ces méthodes en
> enrichissant les entrées : en ajoutant $x_1^2, x_2^2, x_1 x_2$, une frontière linéaire
> dans l'espace augmenté (dimension 5) devient une **conique** dans le plan d'origine.
> Nous l'illustrerons plus bas (figure 4.6).

## 2. Régression linéaire d'une matrice indicatrice (section 4.2)

On code chaque réponse par un vecteur indicateur $y_i = (y_{i1}, \dots, y_{iK})$ avec
$y_{ik} = 1$ si $g_i = k$ et $0$ sinon. On empile ces vecteurs dans la **matrice
indicatrice** $\mathbf{Y}$ de taille $N \times K$ (une seule valeur 1 par ligne).

On ajuste simultanément les $K$ régressions linéaires par moindres carrés
(éq. (4.3) du livre) :
$$\hat{\mathbf{Y}} = \mathbf{X} (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{Y},
\qquad \hat{\mathbf{B}} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{Y},$$
où $\mathbf{X}$ ($N \times (p+1)$) contient une première colonne de 1.

Une nouvelle entrée $x$ est classée en calculant le vecteur ajusté
$\hat f(x)^T = (1, x^T)\hat{\mathbf{B}}$ (de taille $K$) puis
$$\hat G(x) = \operatorname*{arg\,max}_{k} \hat f_k(x) \qquad \text{(éq. (4.4) du livre).}$$

**Justification.** $\hat f_k(x)$ estime $\mathbb{E}(Y_k \mid X = x) = \Pr(G = k \mid X = x)$,
puisque $Y_k$ vaut 1 exactement quand $G = k$. Classer vers la plus grande composante revient
donc à imiter la règle de Bayes.

**Propriété : les composantes ajustées somment à 1.** En effet,
$\hat{\mathbf{Y}} \mathbf{1}_K = \mathbf{H} \mathbf{Y} \mathbf{1}_K$ avec
$\mathbf{H} = \mathbf{X}(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T$ la matrice de projection
orthogonale sur l'espace engendré par les colonnes de $\mathbf{X}$.

- Chaque ligne de $\mathbf{Y}$ contient exactement un 1, donc $\mathbf{Y}\mathbf{1}_K = \mathbf{1}_N$.
- $\mathbf{1}_N$ est la première colonne de $\mathbf{X}$, donc il est laissé fixe par la
  projection : $\mathbf{H}\mathbf{1}_N = \mathbf{1}_N$.

Ainsi $\hat{\mathbf{Y}}\mathbf{1}_K = \mathbf{1}_N$ : pour tout $x$, $\sum_k \hat f_k(x) = 1$.

> **Remarque.** Ce ne sont pas pour autant des probabilités : rien n'empêche $\hat f_k(x)$
> d'être négatif ou supérieur à 1, surtout loin des données (la régression est linéaire et
> donc non bornée).